# 06 · Extending context: PI, NTK-aware and YaRN

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/architecture/06_rope_scaling.ipynb) &nbsp; 📖 Concept deck: [Long context I: stretching RoPE](https://nilesh-salpe.github.io/visual-ai/architecture/07-rope-scaling/) &nbsp; ⏱️ Suggested time: **45 min**

**You will build:** the three standard RoPE-scaling methods as frequency transforms, the YaRN attention temperature, and a check of which dimensions leave the trained angle range.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def rope_freqs(d, base=10000.0):
    """theta_i = base^(-2i/d), i = 0..d/2-1."""
    return base ** (-np.arange(0, d, 2) / d)

## Task 1: Position Interpolation (Chen et al., 2023)

To run a model trained on length $L$ at length $s\cdot L$, squeeze positions: $m \to m/s$. That is the same as dividing **every** frequency by $s$.
All angles then stay inside the range seen in training. The cost: neighbouring tokens become $s\times$ harder to tell apart in the fast dimensions.

In [ ]:
def pi_freqs(d, s, base=10000.0):
    # TODO: rope_freqs(d, base) / s
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_pi_freqs(f):
    th = rope_freqs(128); p = f(128, 4)
    assert np.allclose(p, th / 4), "every frequency divided by s"
    assert np.allclose((4 * 2048) * p, 2048 * th), "the max angle at 4x length must equal the trained max angle"

check("pi_freqs", _test_pi_freqs, pi_freqs)

## Task 2: NTK-aware: change the base instead

Instead of slowing every dimension equally, raise the base: $b' = b\cdot s^{\,d/(d-2)}$ (YaRN eq. 16).
Effect: the **fastest** pair ($i=0$) is unchanged, so local order stays sharp, and the **slowest** pair is slowed by exactly $s$, like PI. Dimensions in between are slowed by a smoothly varying amount.

In [ ]:
def ntk_freqs(d, s, base=10000.0):
    # TODO: rope_freqs(d, base * s ** (d / (d - 2)))
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_ntk_freqs(f):
    th = rope_freqs(128); n = f(128, 8)
    assert np.isclose(n[0], th[0]), "the highest frequency must be unchanged"
    assert np.isclose(n[-1], th[-1] / 8), "the lowest frequency must be divided by exactly s"
    assert np.all(np.diff(th / n) > 0), "the slow-down factor should grow smoothly from fast to slow pairs"

check("ntk_freqs", _test_ntk_freqs, ntk_freqs)

## Task 3: YaRN: NTK-by-parts plus temperature (Peng et al., 2023)

Decide per dimension by how many full turns it makes inside the original context $L$:
$r(i) = L/\lambda_i$, with wavelength $\lambda_i = 2\pi/\theta_i$.
* $r < \alpha$ (fewer than α turns, slow pairs): **interpolate fully**, $\theta/s$, as PI does.
* $r > \beta$ (many turns, fast pairs): **leave alone**.
* In between: blend with the ramp $\gamma(r) = (r-\alpha)/(\beta-\alpha)$: $h(\theta) = (1-\gamma)\,\theta/s + \gamma\,\theta$.

Paper's recommendation for Llama: $\alpha=1$, $\beta=32$. YaRN also scales the attention logits through $\sqrt{1/t} = 0.1\ln s + 1$ (≈ 1.208 at $s=8$), because stretched contexts flatten the softmax.

In [ ]:
def yarn_freqs(d, s, L, base=10000.0, alpha=1.0, beta=32.0):
    # TODO: theta = rope_freqs(d, base); r = L / (2*pi/theta); gamma = clip((r - alpha) / (beta - alpha), 0, 1)
    #       return (1 - gamma) * theta / s + gamma * theta
    raise NotImplementedError

def yarn_mscale(s):
    # TODO: 0.1 * ln(s) + 1
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_yarn_freqs(f):
    th = rope_freqs(128); y = f(128, 8, 4096)
    r = 4096 * th / (2 * np.pi)
    assert np.allclose(y[r > 32], th[r > 32]), "fast pairs (r > beta) must be unchanged"
    assert np.allclose(y[r < 1], th[r < 1] / 8), "slow pairs (r < alpha) must be divided by s"
    mid = (r >= 1) & (r <= 32)
    assert mid.any() and np.all((y[mid] <= th[mid] + 1e-12) & (y[mid] >= th[mid] / 8 - 1e-12)), "middle pairs must be between theta/s and theta"
    ms = yarn_mscale_solution if f.__name__.endswith('_solution') else yarn_mscale
    assert np.isclose(ms(8), 1.2079, atol=1e-4), "yarn_mscale(8) should be about 1.208"

check("yarn_freqs", _test_yarn_freqs, yarn_freqs)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Position Interpolation (Chen et al., 2023)
def pi_freqs_solution(d, s, base=10000.0):
    return rope_freqs(d, base) / s

check("pi_freqs (reference)", _test_pi_freqs, pi_freqs_solution)

In [ ]:
# @title Solution for Task 2: NTK-aware: change the base instead
def ntk_freqs_solution(d, s, base=10000.0):
    return rope_freqs(d, base * s ** (d / (d - 2)))

check("ntk_freqs (reference)", _test_ntk_freqs, ntk_freqs_solution)

In [ ]:
# @title Solution for Task 3: YaRN: NTK-by-parts plus temperature (Peng et al., 2023)
def yarn_freqs_solution(d, s, L, base=10000.0, alpha=1.0, beta=32.0):
    theta = rope_freqs(d, base)
    r = L * theta / (2 * np.pi)
    gamma = np.clip((r - alpha) / (beta - alpha), 0, 1)
    return (1 - gamma) * theta / s + gamma * theta

def yarn_mscale_solution(s):
    return 0.1 * np.log(s) + 1

check("yarn_freqs (reference)", _test_yarn_freqs, yarn_freqs_solution)

---
## Experiments
**Which dimensions go out of distribution?** A model trained at $L = 4096$ has only ever seen angles $m\theta_i \le 4095\,\theta_i$ in pair $i$. For the *slow* pairs that is less than one full turn, so an unseen angle is genuinely new to the model. Run at $s = 8$ (32K tokens) and count, per method, the slow pairs (wavelength > L) whose maximum angle exceeds the trained maximum.

In [ ]:
d, L, s = 128, 4096, 8; th = rope_freqs(d); lam = 2 * np.pi / th; slow = lam > L
methods = {'no scaling (extrapolate)': th, 'PI': pi_freqs_solution(d, s), 'NTK-aware': ntk_freqs_solution(d, s), 'YaRN (by-parts)': yarn_freqs_solution(d, s, L)}
for name, f in methods.items():
    ood = np.sum(slow & ((s * L - 1) * f > L * th))
    fast_res = th[0] / f[0]
    print(f"{name:26} slow pairs pushed past trained angles: {ood:2d}/{slow.sum()}   fastest pair slowed by {fast_res:.2f}x")
print(f"\nYaRN logit scale at s={s}: (0.1 ln s + 1)^2 = {yarn_mscale_solution(s) ** 2:.3f}  (applied as sqrt to both q and k)")

---
## Takeaways
* NTK-aware still pushes most slow pairs slightly past the trained range (only the very slowest pair is slowed by exactly $s$). This is the partial extrapolation YaRN's "by-parts" ramp removes.
* Without scaling, the slow RoPE pairs reach angles never seen in training, and attention scores blow up (PI's analysis: the interpolation bound is ~600× smaller than extrapolation's).
* PI fixes this by slowing everything, at the cost of local resolution. NTK-aware keeps the fast pairs. YaRN interpolates only the slow pairs, adds a temperature, and needs about 0.1% of the pretraining data (400 steps) to fine-tune.
* Llama 3 instead trains long directly: RoPE base 500,000, then six context-extension stages from 8K to 128K over about 800B tokens.